# Subqueries and Common Table Expressions (CTEs)

Sometimes, the answer you need from a database requires multiple steps. For example, what if your boss asks: *"Who are the employees making more than the company average?"* To answer this, you first need to calculate the average, and *then* compare everyone's salary to that average. You cannot do this in a single standard query. You need to nest one query inside another. 

We do this using **Subqueries** (queries inside queries) and **CTEs** (temporary result tables). 

Let's set up our Python sandbox with some HR data to practice!

In [4]:
import sqlite3
import pandas as pd

# 1. Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create Departments and Employees tables
cursor.executescript("""
CREATE TABLE Departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT
);

CREATE TABLE Employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT,
    dept_id INTEGER,
    salary DECIMAL(10, 2)
);

INSERT INTO Departments VALUES (1, 'Engineering'), (2, 'Sales'), (3, 'HR');

INSERT INTO Employees VALUES 
(101, 'Alice', 1, 120000),
(102, 'Bob', 1, 95000),
(103, 'Charlie', 2, 85000),
(104, 'David', 2, 105000),
(105, 'Eve', 3, 70000),
(106, 'Frank', 1, 140000);
""")

print("✅ Database ready! HR data loaded.")

✅ Database ready! HR data loaded.


# 1. The Subquery (In the WHERE Clause)
A subquery is just a regular `SELECT` query placed inside parentheses `()`. The database runs the inner query first, figures out the result, and then passes that result to the outer query.

Let's solve the problem: *"Who makes more than the company average?"*

In [5]:
# The inner query calculates the average (~$102,500)
# The outer query filters employees based on that number
query_where_sub = """
SELECT name, salary 
FROM Employees
WHERE salary > (
    SELECT AVG(salary) 
    FROM Employees
);
"""
print("--- Employees Earning Above Company Average ---")
display(pd.read_sql_query(query_where_sub, conn))

--- Employees Earning Above Company Average ---


,name,salary
0,Alice,120000
1,David,105000
2,Frank,140000


# 2. The Subquery (In the FROM Clause)
You can also use a subquery in the `FROM` clause. Instead of querying a real table, you query the *results* of another query. You are essentially creating a temporary table on the fly. 

*(Note: When you do this, you usually need to give the temporary table an alias using `AS` so you can reference it).*

In [6]:
# Let's say we only want to look at the Engineering department, 
# and find the maximum salary among them.
query_from_sub = """
SELECT MAX(salary) as max_engineering_salary
FROM (
    SELECT * FROM Employees 
    WHERE dept_id = 1
) AS EngineeringStaff;
"""
print("\n--- Max Salary in Engineering ---")
display(pd.read_sql_query(query_from_sub, conn))


--- Max Salary in Engineering ---


,max_engineering_salary
0,140000


# 3. Common Table Expressions (CTEs)
Subqueries are powerful, but if you have multiple nested subqueries, your code quickly becomes a nightmare to read. It's often referred to as "spaghetti code."

**Common Table Expressions (CTEs)** fix this. Using the `WITH` keyword, CTEs allow you to define your temporary tables at the *very top* of your query, give them a clean name, and then query them normally at the bottom. 

It does the exact same thing as a subquery, but it is **much easier to read and maintain**. 

In [7]:
# Let's use a CTE to find employees who make more than their specific DEPARTMENT'S average.

query_cte = """
WITH DepartmentAverages AS (
    -- Step 1: Calculate the average salary per department
    SELECT dept_id, AVG(salary) as avg_dept_salary
    FROM Employees
    GROUP BY dept_id
)

-- Step 2: Join our original table to our temporary CTE table
SELECT 
    e.name, 
    d.dept_name, 
    e.salary, 
    da.avg_dept_salary
FROM Employees e
JOIN Departments d ON e.dept_id = d.dept_id
JOIN DepartmentAverages da ON e.dept_id = da.dept_id
WHERE e.salary > da.avg_dept_salary;
"""
print("\n--- Employees Earning Above Their DEPARTMENT'S Average (Using CTE) ---")
display(pd.read_sql_query(query_cte, conn))

# Close the connection
conn.close()


--- Employees Earning Above Their DEPARTMENT'S Average (Using CTE) ---


,name,dept_name,salary,avg_dept_salary
0,Alice,Engineering,120000,118333.333333
1,David,Sales,105000,95000.000000
2,Frank,Engineering,140000,118333.333333


# 4. CTEs vs Subqueries: Which to choose?
* **Use Subqueries** for quick, simple, one-off filters (like the `WHERE` clause example).
* **Use CTEs** for complex logic, multi-step data transformations, or when you need to reference the exact same subquery multiple times in one script. In modern Data Science and Data Engineering, CTEs are generally preferred for their readability.

---

## Real-World Use Case or Analogy:
Think of Subqueries and CTEs like **Cooking a Complex Meal**:

* **Standard Query**: Making toast. You just put bread in the toaster. One step.
* **Subquery**: You need a chopped onion to put into your soup. While cooking the soup (the outer query), you quickly stop, chop an onion on the side (the inner query), and throw it in. 
* **CTE (`WITH` clause)**: This is the professional chef method known as *Mise en place*. Before you even turn on the stove, you chop your onions, mince your garlic, and measure your spices, placing them all in neat little bowls (your CTEs). When it is time to actually cook the meal (your main query at the bottom), everything is organized, clearly labeled, and ready to be combined perfectly. 

---